# Text2SQL Evaluation with RAGAS

Evaluates `llm.text_to_sql.generate_sql` against the golden dataset in `eval/golden_dataset.jsonl`.

Metrics:
- **execution_match** — do the generated and reference SQL return the same rows when run against the live DB?
- **schema_groundedness** (RAGAS `AspectCritic`) — does the generated SQL only reference tables/columns that exist in the schema?
- **answer_correctness** (RAGAS `AspectCritic`) — is the generated SQL semantically equivalent to the reference SQL for the question?

Plain RAGAS `Faithfulness` is not used here — it decomposes the *response* into natural-language statements to check against context, which doesn't work on raw SQL text (verified separately: it scores ~0 even for a correct query). `AspectCritic` (a direct LLM-as-judge yes/no call) is the metric RAGAS exposes for exactly this kind of custom criterion.

In [1]:
import sys, json, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "eval" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from db.db_connection import fetch_df
from db.schema_introspect import get_schema_text
from llm.text_to_sql import generate_sql

from eval.ragas_eval import (
    load_dataset,
    run_generation,
    run_ragas_judges,
    summarize,
    EVAL_DIR,
)

OpenAI API Key loaded successfully
Database schema loaded and cached in memory


## 1. Load the golden dataset

In [2]:
rows = load_dataset(EVAL_DIR / "golden_dataset.jsonl")
print(f"{len(rows)} golden questions")
pd.DataFrame(rows).head(10)

32 golden questions


,id,question,reference_sql,category,tags
0,q001,List all customer names,SELECT customer_names FROM customers;,customers,"[single_table, projection]"
1,q002,How many customers are there?,SELECT COUNT(*) FROM customers;,customers,"[single_table, aggregation]"
2,q003,Show the customer with index 1,SELECT * FROM customers WHERE customer_index = 1;,customers,"[single_table, filter]"
3,q004,Find customers whose name contains 'Group',SELECT * FROM customers WHERE customer_names I...,customers,"[single_table, filter, text_match]"
4,q005,List all product names,SELECT product_name FROM products;,products,"[single_table, projection]"
5,q006,How many products are there?,SELECT COUNT(*) FROM products;,products,"[single_table, aggregation]"
6,q007,Show the product with index 5,SELECT * FROM products WHERE index = 5;,products,"[single_table, filter]"
7,q008,List all cities in Alabama,SELECT name FROM regions WHERE state = 'Alabam...,regions,"[single_table, filter]"
8,q009,What is the population of Birmingham?,SELECT population FROM regions WHERE name = 'B...,regions,"[single_table, filter]"
9,q010,Show the top 5 regions by population,"SELECT name, population FROM regions ORDER BY ...",regions,"[single_table, sort, limit]"


Optionally slice the dataset for a quick smoke run before scoring the full set (each row costs one generation call + one execution + two judge calls).

In [3]:
# rows = rows[:5]  # uncomment to smoke-test on a subset
len(rows)

32

## 2. Generate SQL for every question and execute it against the live DB

In [4]:
schema_text = get_schema_text()
print(schema_text)

Table: 2017_bodgets - 30 rows
Columns: product_name (text), 2017_budgets (numeric)

Table: customers - 175 rows
Columns: customer_index (integer), customer_names (text)

Table: products - 30 rows
Columns: index (integer), product_name (text)

Table: query_cache - 20 rows
Columns: cache_key (text), question (text), sql_query (text), result_json (jsonb), row_count (integer), created_at (timestamp without time zone), embedding_model (text), embedding (USER-DEFINED)

Table: regions - 994 rows
Columns: id (integer), name (text), county (text), state_code (text), state (text), type (text), latitude (numeric), longitude (numeric), area_code (integer), population (integer), households (integer), median_income (numeric), land_area (numeric), water_area (numeric), time_zone (text)

Table: sales_order - 64104 rows
Columns: ordernumber (text), orderdate (date), customer_name_index (integer), channel (text), currency_code (text), warehouse_code (text), delivery_region_index (integer), product_descr

In [5]:
records = run_generation(rows, schema_text)
pd.DataFrame(records)[["id", "question", "generated_sql", "execution_match"]]

,id,question,generated_sql,execution_match
0,q001,List all customer names,SELECT customer_names FROM customers,True
1,q002,How many customers are there?,SELECT COUNT(*) FROM customers;,True
2,q003,Show the customer with index 1,SELECT * FROM customers WHERE customer_index = 1;,True
3,q004,Find customers whose name contains 'Group',SELECT customer_names FROM customers WHERE cus...,False
4,q005,List all product names,SELECT product_name FROM products,True
5,q006,How many products are there?,SELECT COUNT(*) FROM products,True
6,q007,Show the product with index 5,SELECT * FROM products WHERE index = 5;,True
7,q008,List all cities in Alabama,SELECT name FROM regions WHERE state = 'Alabam...,False
8,q009,What is the population of Birmingham?,SELECT population FROM regions WHERE name = 'B...,True
9,q010,Show the top 5 regions by population,SELECT name FROM regions ORDER BY population D...,False


## 3. Score with RAGAS `AspectCritic` judges

In [6]:
df = run_ragas_judges(records, schema_text)
df

Evaluating:   0%|          | 0/64 [00:00<?, ?it/s]

,id,question,reference_sql,category,generated_sql,generation_error,reference_rows,generated_rows,generated_exec_error,execution_match,schema_groundedness,answer_correctness
0,q001,List all customer names,SELECT customer_names FROM customers;,customers,SELECT customer_names FROM customers,None,175,175.0,NaN,True,1,1
1,q002,How many customers are there?,SELECT COUNT(*) FROM customers;,customers,SELECT COUNT(*) FROM customers;,None,1,1.0,NaN,True,1,1
2,q003,Show the customer with index 1,SELECT * FROM customers WHERE customer_index = 1;,customers,SELECT * FROM customers WHERE customer_index = 1;,None,1,1.0,NaN,True,1,1
3,q004,Find customers whose name contains 'Group',SELECT * FROM customers WHERE customer_names I...,customers,SELECT customer_names FROM customers WHERE cus...,None,43,43.0,NaN,False,1,0
4,q005,List all product names,SELECT product_name FROM products;,products,SELECT product_name FROM products,None,30,30.0,NaN,True,1,1
5,q006,How many products are there?,SELECT COUNT(*) FROM products;,products,SELECT COUNT(*) FROM products,None,1,1.0,NaN,True,1,1
6,q007,Show the product with index 5,SELECT * FROM products WHERE index = 5;,products,SELECT * FROM products WHERE index = 5;,None,1,1.0,NaN,True,1,1
7,q008,List all cities in Alabama,SELECT name FROM regions WHERE state = 'Alabam...,regions,SELECT name FROM regions WHERE state = 'Alabam...,None,9,0.0,NaN,False,1,1
8,q009,What is the population of Birmingham?,SELECT population FROM regions WHERE name = 'B...,regions,SELECT population FROM regions WHERE name = 'B...,None,1,1.0,NaN,True,1,1
9,q010,Show the top 5 regions by population,"SELECT name, population FROM regions ORDER BY ...",regions,SELECT name FROM regions ORDER BY population D...,None,5,5.0,NaN,False,1,1


## 4. Summary

In [7]:
summarize(df)


RAGAS Text2SQL Evaluation Summary
Total questions:        32
SQL generated:          32/32
Execution accuracy:     68.8%
Schema groundedness:    87.5%
Answer correctness:     87.5%

Failing / mismatched cases:
  id                                                                                 question  execution_match  answer_correctness
q004                                               Find customers whose name contains 'Group'            False                   0
q008                                                               List all cities in Alabama            False                   1
q010                                                     Show the top 5 regions by population            False                   1
q011                                            List regions with a median income above 50000            False                   1
q014                                            List regions with more than 100000 households            False                   1
q01

## 5. Save results

In [8]:
from datetime import datetime, timezone

results_dir = EVAL_DIR / "results"
results_dir.mkdir(exist_ok=True)
out_path = results_dir / f"eval_notebook_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}.csv"
df.to_csv(out_path, index=False)
print(f"Saved to {out_path}")

Saved to C:\Users\ashee\Desktop\Text2SQL\eval\results\eval_notebook_20260816T162003Z.csv


## 6. Break down by category / tag
Useful once the golden set grows — see which query patterns (joins, aggregations, quoted identifiers) the model struggles with.

In [9]:
if "category" in df.columns:
    display(
        df.groupby("category")[["execution_match", "schema_groundedness", "answer_correctness"]]
        .mean()
        .sort_values("execution_match")
    )

,execution_match,schema_groundedness,answer_correctness
category,,,
budgets,0.000000,0.333333,1.00
regions,0.428571,1.000000,1.00
join,0.500000,0.000000,0.00
customers,0.750000,1.000000,0.75
sales_order,0.900000,1.000000,0.90
products,1.000000,1.000000,1.00
sales_regions,1.000000,1.000000,1.00
